# LaViC Group 8 
# DS8008 Final Project
## Amazon Home Domain | Toronto Metropolitan University
### Kaggle Version

**Team:** Yousef Moustafa, Jason Yu, Jessie Ma  
**Paper:** LaViC Adapting Large Vision-Language Models to Visually-Aware Conversational Recommendation (KDD 2025)

## 1. Environment Setup

In [6]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

True
2
Tesla T4


In [7]:
!git clone https://github.com/yousef-moustafa/ds8008-group8-lavic.git
%cd ds8008-group8-lavic
!git checkout Yousef

Cloning into 'ds8008-group8-lavic'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 143 (delta 52), reused 60 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 54.37 MiB | 26.59 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/kaggle/working/ds8008-group8-lavic
Branch 'Yousef' set up to track remote branch 'Yousef' from 'origin'.
Switched to a new branch 'Yousef'


In [8]:
!pip install -q bitsandbytes>=0.46.1
!pip install -r requirements.txt

In [9]:
import torch, transformers, peft, pytorch_lightning as pl
from PIL import Image
print(f"Torch: {torch.__version__}, Transformers: {transformers.__version__}")
print(f"PEFT: {peft.__version__}, Lightning: {pl.__version__}")
print("All imports successful!")

Torch: 2.10.0+cu128, Transformers: 5.0.0
PEFT: 0.18.1, Lightning: 2.6.1
All imports successful!


## 2. Data Setup

In [10]:
!unzip -q data/item2meta_train.json.zip -d data/
!ls data/amazon_home/

test.jsonl  train.jsonl  valid.jsonl


## 3. Image Download

In [11]:
!cd src && python crawl_home_images.py
!cd src && python download_valid_images.py

Home item IDs: 2971
Exist: 0, Saved: 2969, Failed: 2, Skipped: 13028
Exist: 0, Saved: 512, Failed: 0


In [12]:
# Verify images downloaded
!echo "Train images:" && ls data/train_images/ | wc -l
!echo "Valid images:" && ls data/valid_images/ | wc -l

Train images:
2969
Valid images:
512


## 4. Stage 1: Visual Knowledge Self-Distillation

In [13]:
# Already installed in Setup — uncomment if running Stage 1 in a fresh session
# !pip install -q bitsandbytes>=0.46.1

In [14]:
# !python src/knowledge_distillation_kaggle.py

In [15]:
# !ls -lht /kaggle/working/out_distilled/
# !cat /kaggle/working/out_distilled/val_metrics_epoch_1.txt
# !cat /kaggle/working/out_distilled/val_metrics_epoch_2.txt

In [16]:
# Verify adapter keys (optional debug)
# lora_keys = [k for k in lora_state_dict.keys() if 'lora_' in k]
# print(f"LoRA-specific keys: {len(lora_keys)}")

In [17]:
# Upload vision LoRA adapter to Kaggle dataset for persistence across sessions
# import json
# metadata = {
#     "title": "LaViC Group8 Vision LoRA Adapter",
#     "id": "yousefmoustafaa/lavic-group8-vision-lora-adapter",
#     "licenses": [{"name": "CC0-1.0"}]
# }
# with open("/kaggle/working/out_distilled/vision_lora_adapter_best/dataset-metadata.json", "w") as f:
#     json.dump(metadata, f, indent=2)
# !kaggle datasets create -p /kaggle/working/out_distilled/vision_lora_adapter_best

### ⚠️ Note: Stage 1 Already Completed

Stage 1 training was completed in a prior session (2 epochs, ~6 hours on T4x2).

**Results:**
- Epoch 1: val_loss = 0.5543, PPL = 1.741
- Epoch 2: val_loss = 0.5356, PPL = 1.708

The trained vision LoRA adapter is saved to the Kaggle dataset:
`yousefmoustafaa/lavic-group8-vision-lora-adapter`

**To reproduce:** uncomment and run the cell above.  
**To continue to Stage 2:** add the dataset as input via the Kaggle sidebar and proceed to Section 5.

## 5. Stage 2: Recommendation Prompt Tuning

### 5.1 Run Stage 2 Training
Trains the LLM LoRA adapter for recommendation prompt tuning.  
Saves LoRA weights every 500 steps to `/kaggle/working/out_finetuned/lora_adapter_step*/`.  
If the session dies, re-running this cell will automatically resume from the latest checkpoint.

In [ ]:
!python src/prompt_tuning_kaggle.py

Seed set to 42
[INFO] Loading base model...
config.json: 1.25kB [00:00, 3.15MB/s]
model.safetensors.index.json: 70.2kB [00:00, 134MB/s]
Fetching 4 files: 100%|███████████████████████████| 4/4 [01:00<00:00, 15.05s/it]
Download complete: 100%|███████████████████| 15.1G/15.1G [01:02<00:00, 9.75MB/s]
Loading weights:   0%|                                  | 0/687 [00:00<?, ?it/s]
Loading weights:   0%| | 1/687 [00:00<00:00, 10082.46it/s, Materializing param=m
Loading weights:   0%| | 1/687 [00:00<00:00, 4568.96it/s, Materializing param=mo
Loading weights:   0%| | 2/687 [00:01<08:47,  1.30it/s, Materializing param=mode
Loading weights:   0%| | 2/687 [00:01<08:47,  1.30it/s, Materializing param=lm_h
Loading weights:   0%| | 2/687 [00:01<08:47,  1.30it/s, Materializing param=lm_h
Loading weights:   0%| | 3/687 [00:02<09:55,  1.15it/s, Materializing param=lm_h
Loading weights:   0%| | 3/687 [00:02<09:55,  1.15it/s, Materializing param=mode
Loading weights:   0%| | 3/687 [00:02<09:55,  1.15it/s

## 6. Results

In [1]:
# View final outputs
!ls -lht /kaggle/working/out_finetuned/
!ls -lh /kaggle/working/out_finetuned/trained_lora_adapter/
!cat /kaggle/working/out_finetuned/results_summary.csv

total 700K
drwxr-xr-x 2 root root 4.0K Apr 14 01:24 trained_lora_adapter
-rw-r--r-- 1 root root 685K Apr 14 01:24 test_results_candidates_st.jsonl
-rw-r--r-- 1 root root  145 Apr 14 01:24 results_summary.csv
drwxr-xr-x 2 root root 4.0K Apr 14 01:24 lora_adapter_step500
total 95M
-rw-r--r-- 1 root root  17K Apr 14 01:24 adapter_config.json
-rw-r--r-- 1 root root  95M Apr 14 01:23 adapter_model.safetensors
-rw-r--r-- 1 root root 5.1K Apr 14 01:24 README.md
candidate_type,lr,num_epochs,HR@1,VR,output_file
candidates_st,5e-05,1,0.16624685138539042,0.9650537634408602,test_results_candidates_st.jsonl


## 7. Save Trained Adapter

In [ ]:
# Upload Stage 2 LoRA adapter to Kaggle dataset for persistence across sessions
# Run this once after training completes

# import json, os
# os.makedirs("/kaggle/working/stage2_adapter_upload", exist_ok=True)
# !cp -r /kaggle/working/out_finetuned/trained_lora_adapter /kaggle/working/stage2_adapter_upload/
# metadata = {
#     "title": "LaViC Group8 Stage2 LLM LoRA Adapter",
#     "id": "yousefmoustafaa/lavic-group8-stage2-lora-adapter",
#     "licenses": [{"name": "CC0-1.0"}]
# }
# with open("/kaggle/working/stage2_adapter_upload/dataset-metadata.json", "w") as f:
#     json.dump(metadata, f)
# !kaggle datasets create -p /kaggle/working/stage2_adapter_upload --dir-mode zip